### **Common data problems**

In [ ]:
# 1. "DATA TYPE" Constraint
assert 1+1 == 2                        # This will pass 
assert 1+1 == 3                        # This will not pass 
assert DF['Revenue'].dtype == 'int'    # Verify that Revenue is now an integer (example) 

.str.strip("...")                      # to remove something from a string column
------------------------------------------------------------------------------------------
# 2. "DATA RANGE" Constraint
# example: future sign-ups exist? Setting all in the future to today's date 
import datetime as dt 
DF['DATE2'] = pd.to_datetime(DF['date1']).dt.date
today = dt.date.today()
DF.loc[DF['DATE2'] > today, 'DATE2'] = today
------------------------------------------------------------------------------------------
# 3. "UNIQUENESS" Constraint
# DETECTION-------------------------------------------------------------------------------
column_names = ['var1','var2','var3']  # Column names to check for duplication
duplicates = DF.duplicated(subset = column_names, keep = False)
# keep: Whether to keep first ('first'), last ('last') or all (False) duplicate values
DF[duplicates].sort_values(by = 'varX') # Output duplicate values

# TREATMENTS------------------------------------------------------------------------------
DF.drop_duplicates(inplace = True)                               # 1st OPTION

# Group by column names and produce statistical summaries          2nd OPTION
column_names = ['var1','var2','var3'] 
summaries = {'var2': 'max', 'var3': 'mean'} 
DF = DF.groupby(by = column_names).agg(summaries).reset_index() 

### **Text and categorical data problems**

In [ ]:
# 4. "MEMBERSHIP" Constraint
inconsistent_cat = set(study_data['blood_type']).difference(categories['blood_type']) 
inconsistent_rows = study_data['blood_type'].isin(inconsistent_categories) 
inconsistent_data = study_data[inconsistent_rows]  
# Drop inconsistent categories and get consistent data only
consistent_data = study_data[~inconsistent_rows]
------------------------------------------------------------------------------------------
# 5. "Categorical Variables"
DF['var'] = DF['var'].str.lower().strip()             # for Value consistency
  
labels = ['x', 'y', 'z']                              # Collapsing data into categories(1)
DF['var2'] = pd.qcut(DF['var1'], q = 3, labels = labels)

bins = [0,200000,500000,np.inf]                       # Collapsing data into categories(2)
labels = ['0-200K', '200K-500K', '500K+'] 
DF['var2'] = pd.cut(DF['var1'], bins=bins, labels=labels)

mapping = {'Microsoft':'Desktop', 'Mac':'Desktop',    # Updating categories by Dictionary
           'Linux':'Desktop', 'IOS':'Mobile', 'Android':'Mobile'} 
DF['var'] = DF['var'].replace(mapping) 
devices['operating_system'].unique()
------------------------------------------------------------------------------------------
# 6. "Text Data Cleaning"
DF["var"] = DF["var"].str.replace("...", "!!!")                              # 1
assert DF["var"].str.len().min() > 40                                        # 2
assert phone['Phone number'].str.contains("+|-").any() == False              # 3
phones['Phone number'] = phones['Phone number'].str.replace(r'\D+', '')      # 4

### **Advanced data problems**

In [ ]:
# 7. "UNIFORMITY"
temp_fah = DF.loc[DF['Temperature'] > 40, 'Temperature']                  # Example
temp_cels = (temp_fah - 32) * (5/9) 
DF.loc[DF['Temperature'] > 40, 'Temperature'] = temp_cels 

DF['Birthday'] = pd.to_datetime(DF['Birthday'], errors = 'coerce')        # for date/ time
# Return NA for rows where conversion failed.

DF['Birthday'] = DF['Birthday'].dt.strftime("%d-%m-%Y") 
------------------------------------------------------------------------------------------
# 8. "Cross Field Validation"
sum_classes = flights[['economy_class', 'business_class', 'first_class']].sum(axis = 1) 
passenger_equ = sum_classes == flights['total_passengers'] 
# Find and filter out rows with inconsistent passenger totals 
inconsistent_pass = flights[~passenger_equ] 
consistent_pass = flights[passenger_equ]

import pandas as pd 
import datetime as dt 
DF['Birthday'] = pd.to_datetime(DF['Birthday']) 
today = dt.date.today() 
age_manual = today.year - DF['Birthday'].dt.year  
age_equ = age_manual == users['Age'] 
# Find and filter out rows with inconsistent age 
inconsistent_age = users[~age_equ] 
consistent_age = users[age_equ]
------------------------------------------------------------------------------------------
# 9. "COMPLETENESS" (missingness)
DF.isna().sum() 

import missingno as msno 
import matplotlib.pyplot as plt
msno.matrix(airquality)                        # Visualize missingness  
plt.show() 

missing = DF[DF['var'].isna()]                 # Isolate missing and complete values aside 
complete = DF[~DF['var'].isna()]

DF_dropped = DF.dropna(subset = ['var'])                             # Option 1
DF_imputed = DF.fillna({'var': DF['var'].mean()})                    # Option 2

![My Chart](../images/13.png)

### **Record linkage**

![My Chart](../images/14.png)

![My Chart](../images/15.png)

In [ ]:
# 10. "Comparing Strings"
from thefuzz import fuzz
fuzz.WRatio('Houston Rockets vs Los Angeles Lakers', 'Lakers vs Rockets') 

from thefuzz import process                  # Define string and array of possible matches
string = "Houston Rockets vs Los Angeles Lakers" 
choices = pd.Series(['Rockets vs Lakers', 'Lakers vs Rockets',  'Houson vs Los Angeles',
                     'Heat vs Bulls']) 
process.extract(string, choices, limit = 2)

# VERY IMPORTANT when you have too many variations in categories which remapping them manually isn't scalable
for state in categories['state']:
    # Find potential matches in states with typoes 
    matches = process.extract(state, survey['state'], limit = survey.shape[0])  
    # For each potential match match 
    for potential_match in matches: 
        # If high similarity score 
        if potential_match[1] >= 80: 
            # Replace typo with correct category 
            survey.loc[survey['state'] == potential_match[0], 'state'] = state
------------------------------------------------------------------------------------------
# 11. "Generating Pairs"
# unlike joins, record linkage does not require exact matches between different pairs of data, and instead can find close matches using string similarity.

import recordlinkage  
# Generate the pairs 
pairs = indexer.index(DF1, DF2) 

# Create a Compare object 
compare_cl = recordlinkage.Compare() 

# Find exact matches for pairs of date_of_birth and state 
compare_cl.exact('date_of_birth', 'date_of_birth', label='date_of_birth') 
compare_cl.exact('state', 'state', label='state')  

# Find similar matches for pairs of surname and address_1 using string similarity 
compare_cl.string('surname', 'surname', threshold=0.85, label='surname') 
compare_cl.string('address_1', 'address_1', threshold=0.85, label='address_1')  

# Find matches(values being 1 for a match, and 0 for not a match for each pair rows in DF)
potential_matches = compare_cl.compute(pairs, DF1, DF2)
matches = potential_matches[potential_matches.sum(axis = 1) => 2] # Probable matches

duplicate_rows = matches.index.get_level_values(1)        # Get indices from DF2 only 
duplicate_rows = matches.index.get_level_values("DF2")    # ANOTHER way

# Finding duplicates in DF2 
DF2_duplicates = DF2[DF2.index.isin(duplicate_rows)] 
# Finding new rows in DF2 
DF2_new = DF2[~DF2.index.isin(duplicate_rows)] 
# Link the DataFrames! 
full_census = pd.concat([DF1, DF2_new])